# Episode 8 — HDMI output

The base overlay has a video pipeline in the PL:

**DDR memory (frame) → video DMA → colour conversion → HDMI encoder → monitor**

A frame is just a NumPy array of `height × width × 3` bytes in memory. Python fills it, and
the video DMA reads it 60 times a second and sends it to the monitor. Anything that makes
an image in Python (NumPy, OpenCV, Pillow) can draw on the screen.

Connect an HDMI monitor before running this notebook. The EDGE base overlay also sends the
same picture to the VGA connector P1 (4 bits per colour), so a VGA monitor works too.

In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.video import VideoMode, PIXEL_BGR
import numpy as np
import cv2
import time

base = BaseOverlay("base.bit")
hdmi_out = base.video.hdmi_out
hdmi_out.configure(VideoMode(640, 480, 24), PIXEL_BGR)   # 640x480, 3 bytes per pixel, OpenCV's BGR order
hdmi_out.start();

def show(img):
    """Copy an image (480x640x3, uint8, BGR) to a new frame and send it to the monitor."""
    frame = hdmi_out.newframe()
    frame[:] = img
    hdmi_out.writeframe(frame)

## 1. Solid colours

Pixels are stored **B, G, R**.

In [ ]:
for name, bgr in [("red", (0, 0, 255)), ("green", (0, 255, 0)), ("blue", (255, 0, 0)), ("white", (255, 255, 255))]:
    img = np.zeros((480, 640, 3), np.uint8)
    img[:] = bgr
    show(img)
    print(name)
    time.sleep(1)

## 2. Colour bars with NumPy slicing

In [ ]:
bars = [(255, 255, 255), (0, 255, 255), (255, 255, 0), (0, 255, 0),
        (255, 0, 255), (0, 0, 255), (255, 0, 0), (0, 0, 0)]
img = np.zeros((480, 640, 3), np.uint8)
for i, bgr in enumerate(bars):
    img[:, i * 80:(i + 1) * 80] = bgr
show(img)

## 3. Shapes and text with OpenCV

In [ ]:
img = np.full((480, 640, 3), 30, np.uint8)
cv2.rectangle(img, (40, 40), (600, 440), (255, 255, 255), 2)
cv2.circle(img, (160, 240), 80, (0, 200, 255), -1)
cv2.line(img, (300, 150), (600, 330), (0, 255, 0), 4)
cv2.putText(img, "EDGE ZYNQ", (290, 130), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3)
cv2.putText(img, "PYNQ + HDMI", (300, 400), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 200, 0), 2)
show(img)

## 4. Animation: a bouncing ball

Each loop draws a new picture and sends it. The frame rate is shown on the 7-segment display.
Runs for 10 seconds.

In [ ]:
x, y, dx, dy, r = 100, 100, 7, 5, 30
frames, t0 = 0, time.time()
while time.time() - t0 < 10:
    img = np.zeros((480, 640, 3), np.uint8)
    cv2.circle(img, (x, y), r, (0, 128, 255), -1)
    show(img)
    x, y = x + dx, y + dy
    if not r <= x <= 640 - r:
        dx = -dx
    if not r <= y <= 480 - r:
        dy = -dy
    frames += 1
    if frames % 30 == 0:
        base.seven_seg.show_number(round(frames / (time.time() - t0), 1))
print("{:.1f} frames per second".format(frames / (time.time() - t0)))

## 5. A clock and temperature screen

Updates once a second for 20 seconds.

In [ ]:
for _ in range(20):
    img = np.zeros((480, 640, 3), np.uint8)
    cv2.putText(img, time.strftime("%H:%M:%S"), (90, 220), cv2.FONT_HERSHEY_SIMPLEX, 3.0, (255, 255, 255), 6)
    cv2.putText(img, "Zynq {:.1f} C   LM35 {:.1f} C".format(base.xadc.temperature(), base.xadc.lm35_celsius()),
                (90, 320), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
    show(img)
    time.sleep(1)

## 6. A picture slideshow

Shows the JPEG and PNG pictures it finds in your notebooks folder, fitted to the screen.
Press **button 0** for the next picture; each one also moves on by itself after 3 seconds.
Upload your own photos to this folder to see them.

In [ ]:
import glob, os

def fit(img, w=640, h=480):
    """Resize to fit the screen, keeping the shape, with black borders."""
    scale = min(w / img.shape[1], h / img.shape[0])
    small = cv2.resize(img, (int(img.shape[1] * scale), int(img.shape[0] * scale)))
    out = np.zeros((h, w, 3), np.uint8)
    y0, x0 = (h - small.shape[0]) // 2, (w - small.shape[1]) // 2
    out[y0:y0 + small.shape[0], x0:x0 + small.shape[1]] = small
    return out

files = sorted(f for ext in ("jpg", "JPG", "jpeg", "png")
               for f in glob.glob("/home/xilinx/jupyter_notebooks/**/*." + ext, recursive=True))[:10]
files = files or sorted(glob.glob("*.jpg") + glob.glob("*.png"))
print(len(files), "pictures")

for f in files:
    img = cv2.imread(f)
    if img is None:
        continue
    show(fit(img))
    t0 = time.time()
    while time.time() - t0 < 3 and not base.buttons[0].read():
        time.sleep(0.05)
    while base.buttons[0].read():          # wait for the button to be released
        time.sleep(0.05)

## 7. Higher resolutions

The pipeline also runs at 800×600, 1280×720, 1280×1024 and 1920×1080. Close the output,
configure the new mode, and start it again. Bigger frames take longer to draw in Python.

In [ ]:
hdmi_out.close()
hdmi_out.configure(VideoMode(1280, 720, 24), PIXEL_BGR)
hdmi_out.start();
img = np.zeros((720, 1280, 3), np.uint8)
cv2.putText(img, "1280 x 720", (380, 380), cv2.FONT_HERSHEY_SIMPLEX, 3.0, (255, 255, 255), 6)
frame = hdmi_out.newframe()
frame[:] = img
hdmi_out.writeframe(frame)

## Close the HDMI output

In [ ]:
hdmi_out.close()

## Try it yourself

Draw a bar graph of the four slide switches on the screen (a tall green bar for ON,
a short grey bar for off) and update it ten times a second for 20 seconds.

In [ ]:
# your code here